# as-strided-noncontig-source — ex8: 1-D convolution via as_strided + einsum pipeline

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `as-strided-noncontig-source`. Running the final beacon cell reports progress against the `Numpy: Applied patterns and advanced` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Applied patterns and advanced` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`as-strided-noncontig-source`** (exercise 8). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "as-strided-noncontig-source"
DD_SUBTOPIC = "Numpy: Applied patterns and advanced"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## strides and non-contiguity — quick refresher

**Stride** = number of *elements* (not bytes) to advance one step along an axis. A contiguous `(H, W)` float tensor has stride `(W, 1)`.

**`torch.as_strided(input, size, stride)`** builds a zero-copy view at the exact (shape, stride) you specify. It bypasses safety checks — overlapping windows, out-of-bounds offsets, the works. Powerful, dangerous, and the foundation of rolling-window tricks, im2col, and stride-based broadcasting hacks.

**`.contiguous()`** materializes a row-major copy if the current strides aren't already row-major. Required before `.view()`; optional but often a perf-vs-memory trade-off otherwise.

### Exercise 8 — 1-D convolution via as_strided + einsum pipeline

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Create
> LO: Compose `as_strided` (build the window view) with `einsum` (contract over the window axis) to implement 1-D valid convolution end-to-end.
> Keywords: convolution, as_strided, einsum, rolling-window, integrative
> ```

**KCs targeted:** `as-strided-rolling-window`, `einsum-axis-contract`, `strided-conv-pipeline`

Implement `ex8_conv1d_via_strided(x, kernel)` — a valid 1-D convolution (no padding, stride 1) using **only** `t.as_strided` for the windowing and **only** `t.einsum` for the summation.

Given `x` of shape `(L,)` and `kernel` of shape `(K,)`, return an output of shape `(L - K + 1,)` where `out[i] = sum_j x[i+j] * kernel[j]`.

**Pipeline (build this in order):**
1. Pull `(L,)`, `(sL,)` off `x.shape` / `x.stride()`. Print them.
2. Use `t.as_strided` to construct a windows view of shape `(L - K + 1, K)` with stride `(sL, sL)`. Print the windows shape + stride.
3. Contract the windows with the kernel via `t.einsum('ij,j->i', windows, kernel)`. Print the output shape.

The test compares against `torch.nn.functional.conv1d` for correctness.

> ⚠️ **Integrative.** Three concepts in one pipeline (stride math + window view + einsum reduction). Step through it with print statements — don't try to one-line it on the first attempt.

In [ ]:
def ex8_conv1d_via_strided(x: Tensor, kernel: Tensor) -> Tensor:
    """1-D valid convolution implemented with as_strided + einsum."""
    raise NotImplementedError()


def _test_ex8():
    import torch.nn.functional as F

    # Small hand-checkable case.
    x = t.tensor([1.0, 2.0, 3.0, 4.0, 5.0])
    kernel = t.tensor([1.0, 0.0, -1.0])  # finite-difference filter
    out = ex8_conv1d_via_strided(x, kernel)
    expected = t.tensor([1.0 - 3.0, 2.0 - 4.0, 3.0 - 5.0])  # [-2, -2, -2]
    assert out.shape == (3,), f'expected (3,), got {tuple(out.shape)}'
    assert t.allclose(out, expected), f'finite-diff mismatch: {out} vs {expected}'

    # Compare against torch.nn.functional.conv1d on a larger random case.
    # Note: F.conv1d does cross-correlation (same as our formula), so no flip.
    t.manual_seed(7)
    x2 = t.randn(32)
    k2 = t.randn(5)
    ours = ex8_conv1d_via_strided(x2, k2)
    ref = F.conv1d(x2.view(1, 1, -1), k2.view(1, 1, -1)).view(-1)
    assert ours.shape == ref.shape, f'shape mismatch: {ours.shape} vs {ref.shape}'
    assert t.allclose(ours, ref, atol=1e-5), f'value mismatch vs F.conv1d:\n{ours}\n{ref}'

    # Smoke check: the windows view should share storage with x (no copy).
    L, K = x.shape[0], kernel.shape[0]
    sL, = x.stride()
    windows_dbg = t.as_strided(x, size=(L - K + 1, K), stride=(sL, sL))
    assert windows_dbg.data_ptr() == x.data_ptr(), 'windows view must alias x storage'

    print(f'x.shape={tuple(x.shape)}  x.stride()={x.stride()}')
    print(f'windows.shape={tuple(windows_dbg.shape)}  windows.stride()={windows_dbg.stride()}')
    print(f'out.shape={tuple(out.shape)}  out={out.tolist()}')
    print(f'matches F.conv1d on len-32 input: {t.allclose(ours, ref, atol=1e-5)}')
    _dd_passed.add('ex8')
    print("ex8 ✓")

_test_ex8()

<details><summary>Solution</summary>

```python
def ex8_conv1d_via_strided(x: Tensor, kernel: Tensor) -> Tensor:
    L = x.shape[0]
    K = kernel.shape[0]
    sL, = x.stride()
    windows = t.as_strided(x, size=(L - K + 1, K), stride=(sL, sL))
    return t.einsum('ij,j->i', windows, kernel)
```

**Why this is the canonical "convolution from scratch" trick.** Modern conv kernels under the hood do exactly this — build a windowed view (im2col-style) and reduce via matmul/einsum. The only reason `torch.nn.functional.conv1d` is faster is the fused cuDNN kernel; the math is identical.

**Subtle gotcha.** `F.conv1d` does cross-correlation by default — same formula as ours. "True" convolution flips the kernel: `out[i] = sum_j x[i+j] * kernel[K-1-j]`. Pass `kernel.flip(0)` if you ever need the textbook signal-processing convention.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex8'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex8',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()